# 04_search_demo
Query the FAISS index with new images and visualize results.

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import torch

# Ensure the project root is on PYTHONPATH so local src/ imports work in notebook
project_root = os.path.abspath(os.path.join('..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 1. Đổi import từ clip_embedding sang vit_embedding
from src.vit_embedding import load_vision_model, load_image
from src.vector_search import load_faiss_index, search_index

metadata = pd.read_csv('../data/metadata.csv')
index = load_faiss_index('../data/faiss_index.faiss')

# 2. Sử dụng hàm load model của ViT
model, processor, device = load_vision_model()

def query_image(image_path, top_k=5):
    img = load_image(image_path)
    inputs = processor(images=img, return_tensors='pt').to(device)

    with torch.no_grad():
        # 3. ViT inference: lấy trực tiếp pooler_output (vector 768 chiều)
        outputs = model(**inputs)
        vec_tensor = outputs.pooler_output
        
        # Chuẩn hóa L2 (quan trọng để dùng với Inner Product FAISS Index)
        vec = torch.nn.functional.normalize(vec_tensor, p=2, dim=1).cpu().numpy()[0]

    distances, indices = search_index(index, vec, top_k=top_k)
    return distances, indices

# Select a sample query image from metadata for reliable demo.
query_path = 'F:/25-26 kì 2/chuyên đề thầy Thành/TrafficSignsProject/data_test/Yield_sign_in_Monaco.jpg'
if query_path is None or not os.path.isfile(query_path):
    raise FileNotFoundError('No valid sample image found in metadata. Please run 01_prepare_dataset first.')

print('Using query image:', query_path)
distances, indices = query_image(query_path, top_k=5)
print('Distances:', distances)

# In kết quả cho dễ đọc hơn một chút
for score, idx in zip(distances, indices):
    print(f"{score:.4f} | {metadata.iloc[idx]['label']} | {metadata.iloc[idx]['group']}")

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 399.58it/s, Materializing param=visual_projection.weight]                                
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using query image: F:/25-26 kì 2/chuyên đề thầy Thành/TrafficSignsProject/data_test/Yield_sign_in_Monaco.jpg
Distances [0.8933821  0.88937914 0.88300276 0.8818197  0.880453  ]
0.8933821 W.215b nguy hiểm
0.88937914 W.215c nguy hiểm
0.88300276 W.215b nguy hiểm
0.8818197 W.215a nguy hiểm
0.880453 W.228d nguy hiểm
